# SRE control plane: reproduce the checkout incident

Simulates a checkout regression, then creates the role, skill, tool, and workflow from the article.

Requires Elastic 9.4

`KIBANA_SPACE` must be a space whose solution view is not Elasticsearch, otherwise Observability
Cases is disabled there and the workflow creates a case you cannot open.

In [51]:
%pip install -q elasticsearch==9.4 python-dotenv==1.0.1 requests==2.32.3


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [52]:
import json, os, random, time
from datetime import datetime, timedelta, timezone

import requests
from dotenv import load_dotenv
from elasticsearch import Elasticsearch, helpers

load_dotenv()

ES_URL = os.getenv("ELASTICSEARCH_URL")
API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
KIBANA_URL = os.getenv("KIBANA_URL", "").rstrip("/")
SPACE = os.getenv("KIBANA_SPACE", "default")
BASE = KIBANA_URL if SPACE == "default" else f"{KIBANA_URL}/s/{SPACE}"

headers = {
    "Authorization": f"ApiKey {API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}

es = Elasticsearch(ES_URL, api_key=API_KEY, request_timeout=30)

print(f"{es.info()['version']['number']} | space={SPACE}")

9.4.3 | space=obs


## Create the indices

Single-hyphen names on purpose: `logs-*-*`, `metrics-*-*`, and `traces-*-*` match built-in
templates that create data streams only.

In [53]:
LOGS, TRACES, METRICS = "logs-checkout", "traces-checkout", "metrics-checkout"

MAPPINGS = {
    LOGS: ["service.name", "service.version", "host.name", "log.level", "error.type"],
    TRACES: [
        "trace.id",
        "service.name",
        "service.version",
        "host.name",
        "transaction.name",
        "span.name",
        "event.outcome",
    ],
    METRICS: ["service.name", "service.version", "host.name"],
}
NUMERIC = {
    LOGS: {"http.response.status_code": "integer", "event.duration_ms": "float"},
    TRACES: {"event.duration_ms": "float"},
    METRICS: {
        "db.connection_pool.active": "integer",
        "db.connection_pool.max": "integer",
        "db.connection_pool.wait_ms": "float",
    },
}

for index, keywords in MAPPINGS.items():
    props = {"@timestamp": {"type": "date"}}
    props.update({k: {"type": "keyword"} for k in keywords})
    props.update({k: {"type": t} for k, t in NUMERIC[index].items()})
    if index == LOGS:
        props["message"] = {"type": "text"}

    es.indices.delete(index=index, ignore_unavailable=True)
    es.indices.create(
        index=index,
        mappings={"properties": props},
        settings={"number_of_shards": 1, "number_of_replicas": 0},
    )

print("indices created")

indices created


## Simulate the incident

Deployment `2026.07.09.1` exhausts the connection pool. Timestamps are relative to now so the
data always falls inside a recent search window.

In [54]:
random.seed(42)

SERVICE, HOSTS = "checkout-api", ["checkout-7f4d-a1", "checkout-7f4d-b2"]
HEALTHY, REGRESSED = "2026.07.02.3", "2026.07.09.1"

now = datetime.now(timezone.utc).replace(second=0, microsecond=0)
deploy_at = now - timedelta(minutes=30)
start = now - timedelta(minutes=90)

logs, traces, metrics = [], [], []
minute, n = start, 0

while minute < now:
    bad = minute >= deploy_at
    version = REGRESSED if bad else HEALTHY

    for _ in range(4):
        n += 1
        host = random.choice(HOSTS)
        ts = (minute + timedelta(seconds=random.randint(0, 59))).isoformat()
        failed = bad and random.random() < 0.7
        ms = (
            random.uniform(1900, 2600)
            if failed
            else (random.uniform(600, 1200) if bad else random.uniform(140, 220))
        )

        if failed:
            extra = {
                "log.level": "ERROR",
                "http.response.status_code": 500,
                "error.type": "PoolExhaustedException",
                "message": "Timeout waiting for connection from pool after 2000ms",
            }
        elif bad:
            extra = {
                "log.level": "WARN",
                "http.response.status_code": 200,
                "message": "Checkout completed after slow payment-gateway call",
            }
        else:
            extra = {
                "log.level": "INFO",
                "http.response.status_code": 200,
                "message": "Checkout completed",
            }

        common = {
            "@timestamp": ts,
            "service.name": SERVICE,
            "service.version": version,
            "host.name": host,
            "event.duration_ms": round(ms, 1),
        }
        logs.append({**common, **extra})
        traces.append(
            {
                **common,
                "trace.id": f"{n:08x}",
                "transaction.name": "POST /checkout",
                "span.name": "payment-gateway.charge",
                "event.outcome": "failure" if failed else "success",
            }
        )

    for host in HOSTS:
        metrics.append(
            {
                "@timestamp": minute.isoformat(),
                "service.name": SERVICE,
                "service.version": version,
                "host.name": host,
                "db.connection_pool.active": 20 if bad else random.randint(4, 6),
                "db.connection_pool.max": 20,
                "db.connection_pool.wait_ms": round(
                    random.uniform(1800, 2000) if bad else random.uniform(0, 5), 1
                ),
            }
        )

    minute += timedelta(minutes=1)

for index, docs in ((LOGS, logs), (TRACES, traces), (METRICS, metrics)):
    helpers.bulk(es, [{"_index": index, "_source": d} for d in docs])
    es.indices.refresh(index=index)
    print(f"{index}: {len(docs)}")

logs-checkout: 360
traces-checkout: 360
metrics-checkout: 180


## Verify the incident is discoverable

If these do not show a clear before and after split, the agent has nothing to find.

In [55]:
for query in (
    f'FROM {LOGS} | STATS requests = COUNT(*), errors = COUNT(CASE(log.level == "ERROR", 1, NULL))'
    " BY service.version | SORT service.version",
    f"FROM {TRACES} | STATS p95_ms = ROUND(PERCENTILE(event.duration_ms, 95), 1)"
    " BY service.version | SORT service.version",
    f"FROM {METRICS} | STATS avg_active = ROUND(AVG(db.connection_pool.active), 1),"
    " pool_max = MAX(db.connection_pool.max) BY service.version | SORT service.version",
):
    print(es.esql.query(query=query + " | LIMIT 10", format="txt").body)

   requests    |    errors     |service.version
---------------+---------------+---------------
240            |0              |2026.07.02.3   
120            |90             |2026.07.09.1   

    p95_ms     |service.version
---------------+---------------
216.3          |2026.07.02.3   
2570.0         |2026.07.09.1   

  avg_active   |   pool_max    |service.version
---------------+---------------+---------------
5.0            |20             |2026.07.02.3   
20.0           |20             |2026.07.09.1   



## Create the role, skill, and tool

In [56]:
es.security.put_role(
    name="agent-builder-observability-investigator",
    cluster=["monitor_inference"],
    indices=[
        {
            "names": ["logs-*", "metrics-*", "traces-*"],
            "privileges": ["read", "view_index_metadata"],
        }
    ],
    applications=[
        {
            "application": "kibana-.kibana",
            "privileges": ["feature_agentBuilder.read", "feature_actions.read"],
            "resources": [f"space:{SPACE}"],
        }
    ],
)

skill = {
    "id": "checkout-latency-investigation",
    "name": "Checkout latency investigation",
    "description": "Read-only investigation path for checkout latency and error regressions.",
    "content": """# Checkout latency investigation

Use this skill when an engineer asks why checkout latency, errors, or failed transactions increased.

Work through the investigation in this order:

1. Identify the affected service, environment, and time range.
2. Query traces for the slowest transactions in that window.
3. Query logs for errors from the same service and dependency path.
4. Compare current error and latency rates with the previous healthy window.
5. Return the likely cause, supporting evidence, confidence level, and the next safe action.

Do not recommend a production change unless there is a workflow tool assigned for that action.

If the evidence is incomplete, say what data is missing.
""",
}

tool = {
    "id": "checkout_error_logs",
    "type": "index_search",
    "description": """Use this tool to search checkout service logs for errors in a bounded time range.

Required inputs:
- service_name
- environment
- start_time
- end_time

Return:
- matching log samples
- error counts by message
- affected host and pod names when present
""",
    "configuration": {"pattern": LOGS},
}

for name, payload in (("skills", skill), ("tools", tool)):
    r = requests.post(
        f"{BASE}/api/agent_builder/{name}", headers=headers, json=payload, timeout=30
    )
    print(name, "ok" if r.ok else f"{r.status_code} {r.text[:200]}")

skills 409 {"statusCode":409,"error":"Conflict","message":"Skill with id 'checkout-latency-investigation' already exists.","attributes":{}}
tools 400 {"statusCode":400,"error":"Bad Request","message":"Tool with id checkout_error_logs already exists","attributes":{}}


## Create and run the workflow

In [57]:
WORKFLOW = """
name: obs-labs-checkout-control-plane
description: Checkout regression investigation with Agent Builder and case creation.
enabled: true
tags: ["sre-control-plane", "agent-builder", "workflows"]

triggers:
  - type: manual

inputs:
  - name: service_name
    type: string
    default: "checkout-api"
  - name: alert_summary
    type: string
    default: "Checkout API p95 latency increased above 2s and HTTP 500s rose in the last 15 minutes after deployment 2026.07.09.1."

steps:
  - name: rca_analysis
    type: ai.agent
    agent-id: elastic-ai-agent
    create-conversation: true
    with:
      message: |
        Investigate this checkout incident as an SRE would.

        Service: {{ inputs.service_name }}
        Alert: {{ inputs.alert_summary }}

        Search the available logs, traces, and metrics for this service.
        Compare the window before and after the most recent deployment.

        Return a concise likely cause, supporting evidence, confidence, and next safe action.
        If the evidence is incomplete, say what data is missing.

  - name: case_title
    type: ai.agent
    agent-id: elastic-ai-agent
    with:
      conversation_id: "{{ steps.rca_analysis.output.conversation_id }}"
      message: "Produce a short case title for this incident. Output only the title."

  - name: case_description
    type: ai.agent
    agent-id: elastic-ai-agent
    with:
      conversation_id: "{{ steps.rca_analysis.output.conversation_id }}"
      message: "Produce a concise case description. Output only the description."

  - name: create_case
    type: cases.createCase
    with:
      title: "{{ steps.case_title.output.message }}"
      description: "{{ steps.case_description.output.message }}"
      owner: "observability"
      severity: "medium"
      tags: ["sre-control-plane", "agent-builder", "workflows"]

  - name: add_agent_analysis
    type: cases.addComment
    with:
      case_id: "{{ steps.create_case.output.case.id }}"
      comment: |
        ## Agent Builder RCA

        {{ steps.rca_analysis.output.message }}
"""

wf_headers = {**headers, "x-elastic-internal-origin": "Kibana"}

r = requests.post(
    f"{BASE}/api/workflows",
    headers=wf_headers,
    json={"workflows": [{"yaml": WORKFLOW}]},
    params={"overwrite": "true"},
    timeout=30,
)
r.raise_for_status()
WORKFLOW_ID = r.json()["created"][0]["id"]

r = requests.post(
    f"{BASE}/api/workflows/workflow/{WORKFLOW_ID}/run",
    headers=wf_headers,
    json={"inputs": {}},
    timeout=30,
)
r.raise_for_status()
EXECUTION_ID = r.json()["workflowExecutionId"]

while True:
    execution = requests.get(
        f"{BASE}/api/workflows/executions/{EXECUTION_ID}",
        headers=wf_headers,
        timeout=15,
    ).json()
    if execution.get("status", "").lower() in {
        "completed",
        "failed",
        "cancelled",
        "error",
    }:
        break
    time.sleep(5)

print(execution["status"])
for step in execution.get("stepExecutions", []):
    print(f"  {step.get('stepId'):<20} {step.get('status')}")

completed
  rca_analysis         completed
  case_title           completed
  case_description     completed
  create_case          completed
  add_agent_analysis   completed


## Read the case

In [58]:
r = requests.get(
    f"{BASE}/api/cases/_find",
    headers=headers,
    params={
        "owner": "observability",
        "tags": "sre-control-plane",
        "perPage": 1,
        "sortField": "createdAt",
        "sortOrder": "desc",
    },
    timeout=30,
)
case = r.json()["cases"][0]
print(case["title"], "\n")
print(case["description"], "\n")

# The plain /comments endpoint is deprecated and answers with an error object on 9.4.
r = requests.get(
    f"{BASE}/api/cases/{case['id']}/comments/_find",
    headers=headers,
    params={"perPage": 20},
    timeout=30,
)
comments = r.json().get("comments", []) if r.ok else []

if not comments:
    print(f"no comments returned -> {r.status_code} {r.text[:300]}")

for c in comments:
    print(c.get("comment", "")[:2000])

Checkout API p95 Latency Spike and HTTP 500s from Connection Pool Exhaustion After Deployment 2026.07.09.1 

Following deployment 2026.07.09.1, checkout-api began returning HTTP 500 errors on 76% of requests with p95 latency exceeding 2s. All failures are caused by `PoolExhaustedException`, where requests block for the full 2000ms pool-wait timeout before failing. Even successful requests show elevated latency with warnings about slow payment-gateway calls. Both hosts are equally affected. The prior version (2026.07.02.3) showed zero errors under identical traffic volume, confirming a clean deployment regression. 

## Agent Builder RCA

## Checkout API Incident Investigation

### TL;DR
**Deployment `2026.07.09.1` introduced a connection pool exhaustion bug against the payment-gateway, causing 76% of checkout requests to time out with HTTP 500s and p95 latency exceeding 2s.**

---

## Before vs. After Deployment

| Window | Version | Log Volume | ERROR | WARN | INFO | HTTP 500s |
|---|-

## Cleanup

In [60]:
requests.delete(f"{BASE}/api/workflows/{WORKFLOW_ID}", headers=wf_headers, timeout=30)

r = requests.get(
    f"{BASE}/api/cases/_find",
    headers=headers,
    params={"owner": "observability", "tags": "sre-control-plane", "perPage": 50},
    timeout=30,
)
ids = [c["id"] for c in r.json().get("cases", [])]
if ids:
    requests.delete(
        f"{BASE}/api/cases",
        headers=headers,
        params={"ids": json.dumps(ids)},
        timeout=30,
    )

for index in (LOGS, TRACES, METRICS):
    es.indices.delete(index=index, ignore_unavailable=True)

es.options(ignore_status=404).security.delete_role(
    name="agent-builder-observability-investigator"
)
print("cleaned up")

cleaned up
